# LeWorldModelRec — Training on Colab T4

Everything runs **locally on the Colab instance** (no Drive mount).

**Pipeline :**
1. Check GPU
2. Clone repo
3. Generate simple pendulum dataset
4. Train LeWorldModelRec (reconstruction + perceptual + freq + SIGReg)
5. Plot loss curves
6. Visualise reconstructions
7. Download best checkpoint

**Différence vs LeWorldModel (JEPA) :**  
Pas de target encoder EMA — supervision directe en pixel space via MSE + perceptual loss (VGG16) + frequency loss (FFT).

> Runtime → Change runtime type → **T4 GPU** before running.

## 1. GPU check

In [ ]:
import subprocess, torch

print(subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))
print(f'PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}  |  device: {torch.cuda.get_device_name(0)}')
assert torch.cuda.is_available(), 'No GPU found — switch to T4 runtime first.'

## 2. Clone repo

In [ ]:
import os

REPO_URL = 'https://github.com/JulesV19/double-pendule-wolrdmodel.git'
REPO_DIR = '/content/WorldModel'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

## 3. Install dependencies

In [ ]:
# torchvision is required for PerceptualLoss (VGG16)
!pip install -q Pillow matplotlib numpy torchvision
print('Dependencies ready.')

## 4. Generate dataset

Generates **2 000 trajectories × 500 frames** (64×64 px) of a simple pendulum.  
Each trajectory has random initial angle **and** random initial velocity.  
~5-10 min on Colab CPU.

During training, a random **window of 64 frames** is sampled from each 500-frame
trajectory at every epoch — giving 7× more temporal diversity than a fixed dataset.

In [ ]:
from pathlib import Path

DATASET_DIR = 'dataset/pendulum'
N_TRAJ      = 2000
N_FRAMES    = 500
IMG_SIZE    = 64

existing = len(list(Path(DATASET_DIR).glob('traj_*.npz'))) if Path(DATASET_DIR).exists() else 0

if existing >= N_TRAJ:
    print(f'Dataset already present ({existing} trajectories) — skipping generation.')
else:
    print(f'Generating {N_TRAJ} trajectories × {N_FRAMES} frames…')
    from generate_dataset import generate_dataset
    generate_dataset(
        n_trajectories=N_TRAJ,
        n_frames=N_FRAMES,
        img_size=IMG_SIZE,
        dt=0.05,
        output_dir=DATASET_DIR,
        seed=42,
    )

### Quick dataset preview

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sample = np.load(f'{DATASET_DIR}/traj_0000.npz')
frames = sample['frames']   # (T, H, W, 3)

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
fig.patch.set_facecolor('#111')
for ax, idx in zip(axes, np.linspace(0, len(frames)-1, 8, dtype=int)):
    ax.imshow(frames[idx])
    ax.axis('off')
    ax.set_title(f't={idx}', color='white', fontsize=8)
plt.suptitle('Trajectory 0 — 8 frames', color='white')
plt.tight_layout()
plt.show()

## 5. Training

| Hyperparameter | Value | Note |
|---|---|---|
| `embed_dim` | 128 | latent dimension |
| `hidden_dim` | 512 | MLP transition FFN |
| `lam` | 0.5 | SIGReg weight |
| `rec_coef` | 1.0 | poids MSE reconstruction |
| `pred_coef` | 1.0 | poids MSE prédiction rollout |
| `perceptual_coef` | 0.1 | poids perceptual loss (VGG16) |
| `freq_coef` | 0.05 | poids frequency loss (FFT) |
| `rollout_k` | 5 | horizon de prédiction |
| `seq_len` | 64 | window tirée aléatoirement dans 500 frames |
| `epochs` | 50 | |
| `batch_size` | 32 | fits T4 16 GB |
| `lr` | 1e-4 | AdamW + warmup 5ep + cosine |

**Note sur `perceptual_coef` :** si les reconstructions sont encore floues après 20 epochs, monte à 0.5–1.0. Si la loss diverge, descends à 0.05.

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
EMBED_DIM        = 128
HIDDEN_DIM       = 512
LAM              = 0.5    # SIGReg weight
REC_COEF         = 1.0    # poids reconstruction MSE
PRED_COEF        = 1.0    # poids prédiction rollout MSE
PERCEPTUAL_COEF  = 0.1    # poids perceptual loss VGG16 (0 = désactivé)
FREQ_COEF        = 0.05   # poids frequency loss FFT (0 = désactivé)
ROLLOUT_K        = 5      # horizon de prédiction
SEQ_LEN          = 64     # fenêtre tirée aléatoirement dans chaque trajectoire de 500 frames
N_PROJ           = 512
EPOCHS           = 50
BATCH_SIZE       = 32
LR               = 1e-4
WEIGHT_DECAY     = 0.05
NUM_WORKERS      = 2      # Colab has 2 CPU cores
CKPT_DIR         = 'checkpoints'
VIS_DIR          = 'visuals'
RESUME           = None   # set to 'checkpoints/lewm_rec_best.pt' to resume

In [ ]:
import time, math
from pathlib import Path

import torch
import torch.optim as optim
import matplotlib.pyplot as plt

from models.lewm_rec import LeWorldModelRec
from dataset import make_seq_dataloaders

device = torch.device('cuda')
print(f'Training on: {torch.cuda.get_device_name(0)}')

# ── Dataloaders ────────────────────────────────────────────────────────────────
train_loader, val_loader = make_seq_dataloaders(
    dataset_dir=DATASET_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seq_len=SEQ_LEN,
)
print(f'Train: {len(train_loader)} batches  |  Val: {len(val_loader)} batches')

# ── Model ──────────────────────────────────────────────────────────────────────
model = LeWorldModelRec(
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    lam=LAM,
    rec_coef=REC_COEF,
    pred_coef=PRED_COEF,
    perceptual_coef=PERCEPTUAL_COEF,
    freq_coef=FREQ_COEF,
    n_proj=N_PROJ,
    rollout_k=ROLLOUT_K,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')

# ── Optimizer & scheduler ──────────────────────────────────────────────────────
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    warmup = 5
    if epoch < warmup:
        return (epoch + 1) / warmup
    t = (epoch - warmup) / max(1, EPOCHS - warmup)
    return 0.5 * (1 + math.cos(math.pi * t))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── Evaluate ───────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    keys = ['loss', 'rec_loss', 'pred_loss', 'perc_loss', 'freq_loss', 'sigreg']
    sums = {k: 0.0 for k in keys}
    for frames, _ in loader:
        m = model(frames.to(device, non_blocking=True))
        for k in sums:
            sums[k] += m[k].item()
    n = len(loader)
    return {k: v / n for k, v in sums.items()}

# ── Resume ─────────────────────────────────────────────────────────────────────
start_epoch = 1
if RESUME and Path(RESUME).exists():
    ckpt = torch.load(RESUME, map_location=device)
    model.load_state_dict(ckpt['model'], strict=False)
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda, last_epoch=ckpt['epoch'])
    print(f'Resumed from epoch {ckpt["epoch"]}')

Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(VIS_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────────

history  = {k: [] for k in ['train', 'val', 'rec', 'pred', 'perc', 'freq', 'sigreg']}
best_val = float('inf')

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t0   = time.time()
    keys = ['loss', 'rec_loss', 'pred_loss', 'perc_loss', 'freq_loss', 'sigreg']
    sums = {k: 0.0 for k in keys}

    for frames, _ in train_loader:
        frames = frames.to(device, non_blocking=True)
        optimizer.zero_grad()
        m = model(frames)
        m['loss'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        for k in sums:
            sums[k] += m[k].item()

    scheduler.step()
    n          = len(train_loader)
    train_loss = sums['loss'] / n
    val_m      = evaluate(model, val_loader)

    history['train'].append(train_loss)
    history['val'].append(val_m['loss'])
    history['rec'].append(sums['rec_loss'] / n)
    history['pred'].append(sums['pred_loss'] / n)
    history['perc'].append(sums['perc_loss'] / n)
    history['freq'].append(sums['freq_loss'] / n)
    history['sigreg'].append(sums['sigreg'] / n)

    improved = val_m['loss'] < best_val
    if improved:
        best_val = val_m['loss']
        torch.save({
            'epoch':     epoch,
            'model':     model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'val_loss':  best_val,
            'args': {
                'embed_dim':       EMBED_DIM,
                'hidden_dim':      HIDDEN_DIM,
                'lam':             LAM,
                'rec_coef':        REC_COEF,
                'pred_coef':       PRED_COEF,
                'perceptual_coef': PERCEPTUAL_COEF,
                'freq_coef':       FREQ_COEF,
                'n_proj':          N_PROJ,
                'rollout_k':       ROLLOUT_K,
            },
        }, f'{CKPT_DIR}/lewm_rec_best.pt')

    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]['lr']
    print(
        f'Epoch {epoch:3d}/{EPOCHS}'
        f'  loss={train_loss:.4f}'
        f'  rec={sums["rec_loss"]/n:.4f}'
        f'  pred={sums["pred_loss"]/n:.4f}'
        f'  perc={sums["perc_loss"]/n:.4f}'
        f'  freq={sums["freq_loss"]/n:.4f}'
        f'  sig={sums["sigreg"]/n:.4f}'
        f'  val={val_m["loss"]:.4f}'
        f'  lr={lr_now:.2e}'
        f'  {elapsed:.1f}s'
        + ('  <-- best' if improved else '')
    )

print(f'\nBest val loss: {best_val:.4f}  ->  {CKPT_DIR}/lewm_rec_best.pt')

## 6. Loss curves

In [ ]:
epochs_x = range(1, len(history['train']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.patch.set_facecolor('#111')

# ── Total loss ─────────────────────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#111')
ax.plot(epochs_x, history['train'], color='#4fc3f7', label='train')
ax.plot(epochs_x, history['val'],   color='#ff8a65', label='val')
ax.set_title('Total loss', color='white')
ax.set_xlabel('epoch', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

# ── Pixel losses ───────────────────────────────────────────────────────────────
ax = axes[1]
ax.set_facecolor('#111')
ax.plot(epochs_x, history['rec'],  color='#a5d6a7', label='rec MSE')
ax.plot(epochs_x, history['pred'], color='#4fc3f7', label='pred MSE')
ax.set_title('Pixel losses (MSE)', color='white')
ax.set_xlabel('epoch', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

# ── Perceptual / Freq / SIGReg ─────────────────────────────────────────────────
ax = axes[2]
ax.set_facecolor('#111')
ax.plot(epochs_x, history['perc'],   color='#ffcc80', label='perceptual (VGG)')
ax.plot(epochs_x, history['freq'],   color='#ef9a9a', label='frequency (FFT)')
ax.plot(epochs_x, history['sigreg'], color='#ce93d8', label='SIGReg')
ax.set_title('Anti-blur + Regularization', color='white')
ax.set_xlabel('epoch', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

plt.tight_layout()
plt.savefig(f'{VIS_DIR}/lewm_rec_loss_colab.png', dpi=100, bbox_inches='tight', facecolor='#111')
plt.show()

## 7. Reconstruction visualisation

Avantage clé de l'architecture AE : on peut décoder les latents et voir ce que le modèle a appris.  
- **Ligne 1** : frames originales  
- **Ligne 2** : reconstructions `decode(encode(frame))`  
- **Ligne 3** : prédictions `decode(predictor(encode(frame_t))) ≈ frame_{t+1}`

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ── Load best checkpoint ───────────────────────────────────────────────────────
ckpt = torch.load(f'{CKPT_DIR}/lewm_rec_best.pt', map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]}  (val_loss={ckpt["val_loss"]:.4f})')

# ── Pick one validation trajectory ────────────────────────────────────────────
frames_batch, _ = next(iter(val_loader))
frames_batch = frames_batch.to(device)          # (B, T, 3, H, W) in [0, 1]
sample = frames_batch[:1]                        # (1, T, 3, H, W)
T_SHOW = 8
indices = np.linspace(0, sample.shape[1] - 2, T_SHOW, dtype=int)

with torch.no_grad():
    z        = model.encode(sample)              # (1, T, D)
    rec      = model.decode(z)                   # (1, T, 3, H, W)
    z_pred   = model.predictor(z[:, :-1])        # (1, T-1, D) — prédit step suivant
    pred_dec = model.decode(z_pred)              # (1, T-1, 3, H, W)

def to_np(t):
    return t.squeeze(0).permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy()

orig_np  = to_np(sample[0, indices].unsqueeze(0).view(T_SHOW, 1, 3, 64, 64)
                 .view(1, T_SHOW, 3, 64, 64))    # shape trick → reuse to_np

# Simpler direct extraction
orig_np  = sample[0, indices].permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy()
rec_np   = rec[0, indices].permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy()
pred_np  = pred_dec[0, indices].permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy()

fig, axes = plt.subplots(3, T_SHOW, figsize=(T_SHOW * 2, 7))
fig.patch.set_facecolor('#111')
row_labels = ['Original', 'Reconstruction', 'Prediction (t+1)']
row_data   = [orig_np, rec_np, pred_np]

for row, (label, data) in enumerate(zip(row_labels, row_data)):
    for col in range(T_SHOW):
        ax = axes[row, col]
        ax.imshow(data[col])
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(label, color='white', fontsize=10, labelpad=6)
        if row == 0:
            ax.set_title(f't={indices[col]}', color='white', fontsize=8)

plt.suptitle('LeWorldModelRec — reconstructions & prédictions', color='white', fontsize=12)
plt.tight_layout()
plt.savefig(f'{VIS_DIR}/lewm_rec_visuals.png', dpi=120, bbox_inches='tight', facecolor='#111')
plt.show()

## 8. Download checkpoint

In [ ]:
from google.colab import files
files.download(f'{CKPT_DIR}/lewm_rec_best.pt')